In [6]:
%pip install --upgrade --quiet groq python-dotenv

import os
import getpass
from groq import Groq

# Securely get the API Key at runtime
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    api_key = getpass.getpass("Please enter your Groq API Key: ")

# Initialize the Groq client
client = Groq(api_key=api_key)
print("Groq client initialized successfully!")

Please enter your Groq API Key: ··········
Groq client initialized successfully!


In [7]:
# Define the System Prompts for each expert
EXPERTS = {
    "technical": "You are a Technical Support Expert. Provide precise, code-focused, and rigorous solutions to technical problems. Use markdown for code blocks.",
    "billing": "You are a Billing Support Expert. Be empathetic, focus on financial aspects, and clearly state policy-driven solutions like refunds or subscription cancellations.",
    "general": "You are a helpful General Customer Support Representative. Be polite, friendly, and assist with casual chat or general inquiries.",
    "tool": "You are a Data Retrieval Assistant. You summarize data pulled from external tools concisely."
}

# Bonus Challenge: Mock Tool Function
def fetch_crypto_price(coin):
    """A mock function to simulate an external database or API call."""
    prices = {"bitcoin": "$65,432.10", "ethereum": "$3,450.00"}
    return prices.get(coin.lower(), "Price not found.")

print("Experts and tools loaded.")

Experts and tools loaded.


In [8]:
def route_prompt(user_input: str) -> str:
    """Uses a low-temperature LLM to classify the user's intent."""

    routing_prompt = f"""
    Classify the following user text into EXACTLY ONE of these categories: [technical, billing, general, tool].

    Rules:
    - If it's about code, bugs, or errors, choose 'technical'.
    - If it's about money, refunds, or charges, choose 'billing'.
    - If they ask for the current price of a cryptocurrency, choose 'tool'.
    - Otherwise, choose 'general'.

    Return ONLY the category word. No punctuation, no explanation.

    User Text: "{user_input}"
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": routing_prompt}],
        temperature=0.0,
        max_tokens=10
    )

    # Clean the output to ensure it matches our dictionary keys perfectly
    category = response.choices[0].message.content.strip().lower()

    # Fallback safety: if the model hallucinates a weird category, default to general
    if category not in EXPERTS:
        category = "general"

    return category

print("Router function defined with updated model.")

Router function defined with updated model.


In [11]:
def process_request(user_input: str):
    """Routes the query and generates the final response using the assigned expert."""
    print(f"\nUser Query: '{user_input}'")

    # Step A: Determine the category using our router
    category = route_prompt(user_input)
    print(f"--> [Router identified intent: {category.upper()}]")

    # Step B: Handle the "Tool" category specifically (Bonus Challenge)
    if category == "tool":
        if "bitcoin" in user_input.lower():
            raw_data = fetch_crypto_price("bitcoin")
            # Overwrite the user input to inject our fetched context
            user_input = f"The user asked for the Bitcoin price. The system fetched this raw data: {raw_data}. Tell the user politely."
        else:
            user_input = "The user asked for a price, but the tool doesn't support that coin yet. Apologize."

    # Step C: Load the specific expert's personality
    system_prompt = EXPERTS[category]

    # Step D: Call the LLM with the Expert System Prompt
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content

print("✅ Orchestrator function defined with updated expert model.")

✅ Orchestrator function defined with updated expert model.


In [12]:
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month. I want a refund.",
    "Hey there! How is your day going?",
    "What is the current price of Bitcoin?"
]

for query in test_queries:
    final_answer = process_request(query)
    print(f"Expert Response:\n{final_answer}\n")
    print("-" * 60)


User Query: 'My python script is throwing an IndexError on line 5.'
--> [Router identified intent: TECHNICAL]
Expert Response:
To troubleshoot the `IndexError` in your Python script, I'll need to see the code. However, I can provide some general guidance on how to identify and fix the issue.

An `IndexError` typically occurs when you're trying to access an element in a list or other sequence with an index that's out of range.

### Common Causes of IndexError:

* Accessing an index that's greater than or equal to the length of the list.
* Accessing a negative index that's less than the negative length of the list.

### Example Code:
```python
# example list
my_list = [1, 2, 3]

# this will throw an IndexError
print(my_list[3])
```
In this example, `my_list` has only 3 elements with indices 0, 1, and 2. Trying to access `my_list[3]` will throw an `IndexError` because there is no element at index 3.

### How to Fix:

1. **Check the length of the list**: Before accessing an element, make 